# Global stats visualizer (ALL experiments)

Companion to `stats_visualizer.ipynb`. That one is the **paper/revision**
view (4 main datasets, the paper minimizers). THIS one is the **global**
view: every result on disk - all datasets (incl. aids/bzr/enzymes/
proteins/colors/tcr variants), every generator, every minimizer
(including the BLS variants: trainable / ponderation / net /
net-trainable), and the legacy thesis runs.

One table: per (dataset, generator, minimizer), mean and global std for
GED / FED / OC, plus correctness. No dataset or minimizer is filtered out.
Seeds (when present) are pooled into the global std (this view is not about
seed-stability; use the paper notebook's Table B for that).

## 1. Setup + load both stores

In [ ]:
import os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import jsonpickle

module_path = os.path.abspath(os.path.join('..'))
sys.path.insert(0, module_path); os.chdir(module_path)

RESULTS_ROOT   = Path('lab/output/results')
RESULTS_LEGACY = Path('lab/output/results-legacy')
LEGACY_SEED    = -1
METRICS = ['GraphEditDistance', 'FeatureEditDistance', 'OracleCalls', 'Correctness']
PRETTY  = {'GraphEditDistance':'GED','FeatureEditDistance':'FED','OracleCalls':'OC','Correctness':'Corr'}
DEC     = {'GraphEditDistance':2,'FeatureEditDistance':2,'OracleCalls':1,'Correctness':3}
# attributed datasets (FED meaningful): everything with real node features.
ATTRIBUTED = {'synthie', 'bbbp', 'enzymes', 'bzr', 'aids', 'proteins'}

# Global scope grammar: any dataset, any generator token, any minimizer token.
SCOPE_RE = re.compile(r'^(?P<ds>.+?)_(?P<gen>[a-z0-9]+)_(?P<min>[a-z0-9-]+?)(?:_seed(?P<seed>\d+))?$')

def parse_scope(scope, legacy=False):
    m = SCOPE_RE.match(scope)
    if not m: return None
    seed = LEGACY_SEED if legacy else (int(m['seed']) if m['seed'] is not None else 0)
    return m['ds'], m['gen'], m['min'], seed

def _as_float(v):
    try: return float(v)
    except (TypeError, ValueError): return None

def load_store(root, legacy):
    rows=[]; n=0
    if not root.is_dir(): return rows,n
    for f in root.rglob('results_*.json'):
        mm=re.match(r'results_(\d+)_(\d+)\.json', f.name)
        if not mm: continue
        n+=1; fold=int(mm.group(1))
        try:
            data=jsonpickle.decode(f.read_text()); scope=data['config']['scope']
        except Exception: continue
        ps=parse_scope(scope, legacy)
        if ps is None: continue
        ds,gen,mnk,seed=ps
        for sk,entries in data.get('results',{}).items():
            metric=sk.rsplit('.',1)[-1]
            if metric not in METRICS: continue
            for e in entries:
                fv=_as_float(e.get('value'))
                if fv is not None:
                    rows.append({'scope':scope,'fold':fold,'inst_id':str(e.get('id')),
                                 'metric':metric,'value':fv,'ds':ds,'gen':gen,
                                 'min_kind':mnk,'seed':seed,'source':'legacy' if legacy else 'results'})
    return rows,n

rm,nm=load_store(RESULTS_ROOT,False); rl,nl=load_store(RESULTS_LEGACY,True)
print(f"results/ {nm} files -> {len(rm)} rows ; results-legacy/ {nl} files -> {len(rl)} rows")
df=pd.DataFrame(rm+rl)
# per-instance correctness flag (GED/FED averaged only over valid CFs)
_c=(df[df['metric']=='Correctness'][['scope','fold','inst_id','value']]
     .rename(columns={'value':'is_correct'}))
_c['is_correct']=_c['is_correct']>0.5
df=df.merge(_c,on=['scope','fold','inst_id'],how='left'); df['is_correct']=df['is_correct'].fillna(True)
print(f"datasets: {sorted(df['ds'].unique())}")
print(f"generators: {sorted(df['gen'].unique())}")
print(f"minimizers: {sorted(df['min_kind'].unique())}")

## 2. Global table - every (dataset, generator, minimizer)

mean and global std pooled over **all** folds/seeds/legacy for each cell.

In [ ]:
CF_FILTER={'GraphEditDistance','FeatureEditDistance'}
mask=df['metric'].isin(CF_FILTER) & (~df['is_correct'])
df_eff=df[~mask]
per_fold=(df_eff.groupby(['ds','gen','min_kind','seed','fold','metric'])['value'].mean().reset_index())
agg=(per_fold.groupby(['ds','gen','min_kind','metric'])['value']
            .agg(mean='mean', std='std', n='count').reset_index())

def fmt(m,s,d):
    if pd.isna(m): return '-'
    return f"{m:.{d}f} ± {s:.{d}f}" if not pd.isna(s) else f"{m:.{d}f}"

def global_table(agg):
    rows=[]
    for (ds,gen,mn),g in agg.groupby(['ds','gen','min_kind'],sort=True):
        r={'ds':ds,'gen':gen,'min':mn}
        for metric in METRICS:
            row=g[g['metric']==metric]; d=DEC[metric]
            r[PRETTY[metric]]= fmt(row['mean'].iloc[0],row['std'].iloc[0],d) if not row.empty else '-'
        r['n_runs']=int(g['n'].max())
        rows.append(r)
    out=pd.DataFrame(rows)
    if not out.empty and 'FED' in out.columns:
        out.loc[~out['ds'].isin(ATTRIBUTED),'FED']='-'
    return out

tbl=global_table(agg)
print(f"{len(tbl)} (dataset, generator, minimizer) cells")
pd.set_option('display.max_rows', 300)
tbl

## 3. Save global CSV

In [ ]:
out_csv='lab/stats/results_global.csv'
os.makedirs(os.path.dirname(out_csv),exist_ok=True)
tbl.to_csv(out_csv,index=False)
print(f"saved {len(tbl)} rows to {out_csv}")